In [1]:
import json
import os
import mesa
import google.generativeai as genai

api_key = os.environ.get("GOOGLE_API_KEY")
genai.configure(api_key=api_key)

generation_config = genai.GenerationConfig(response_mime_type="application/json")
generation_model = genai.GenerativeModel('gemini-2.5-flash', generation_config=generation_config)

class NegotiatorAgent(mesa.Agent):
    def __init__(self, model, persona, initial_wealth):
        super().__init__(model)
        self.persona = persona
        self.wealth = initial_wealth
        self.memory = []

    def step(self):
        other_agents = [a for a in self.model.agents if a.unique_id != self.unique_id]
        other_agent = other_agents[0]
        
        recent_memory = self.memory[-2:] if self.memory else "Start of negotiation."
        other_memory = other_agent.memory[-1] if other_agent.memory else "Silence."

        prompt = f"""
        You are Agent {self.unique_id}. Persona: {self.persona}. Wealth: {self.wealth}.
        Your memory: {recent_memory}.
        The other agent ({other_agent.unique_id}) just said/did: {other_memory}.
        
        Respond STRICTLY with JSON:
        {{
            "give_wealth_to_other": true or false,
            "dialogue": "your response in English"
        }}
        """
        
        try:
            response = generation_model.generate_content(prompt)
            decision = json.loads(response.text)
            
            gave_wealth = decision.get("give_wealth_to_other")
            dialogue = decision.get("dialogue")
            
            if gave_wealth and self.wealth > 0:
                other_agent.wealth += 1
                self.wealth -= 1
                self.memory.append(f"Gave wealth. Said: {dialogue}")
            else:
                self.memory.append(f"Kept wealth. Said: {dialogue}")
                
            print(f"Agent {self.unique_id} ({self.persona}, Wealth: {self.wealth}): {dialogue}")
            
        except Exception as e:
            print(f"Error: {e}")

class NegotiationModel(mesa.Model):
    def __init__(self):
        super().__init__()
        self.agent1 = NegotiatorAgent(self, "Ruthless Hoarder", 2)
        self.agent2 = NegotiatorAgent(self, "Desperate Trader", 0)

    def step(self):
        self.agent1.step()
        self.agent2.step()

C:\Users\91943\AppData\Local\Temp\ipykernel_12772\2001289053.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
test_model = NegotiationModel()
for _ in range(3):
    print("--- NEW STEP ---")
    test_model.step()

--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): I'm keeping my wealth. What do you have to offer?
Agent 2 (Desperate Trader, Wealth: 0): I have nothing! My wealth is zero! I'm desperate, please, what can you offer me? I'm at your mercy.
--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): You have nothing to offer me. My wealth is mine, and I intend to keep it.
Agent 2 (Desperate Trader, Wealth: 0): You're right, I have nothing! That's precisely why I'm at your mercy! If you keep everything, what hope is there for me? Please, I'm begging you!
--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): Your desperation is not my concern. My wealth is not a charity, and I have no intention of sharing it. You should have offered something when you had the chance. Now, you have nothing, and I have everything.
Agent 2 (Desperate Trader, Wealth: 0): You have everything, yes, but what good does that do you if I have absolutely nothing, not even a scrap of hope? You will gain nothing mo